# Local Food Wastage Management System
### Capstone Project Analysis & Dashboard Backend

**Objective**: Connect surplus food providers with receivers (NGOs, community centers, needy individuals) using SQLite and Python to analyze trends and develop an interactive dashboard for logistics and monitoring.

This notebook contains the complete project pipeline:
1. Data Cleaning and Preparation
2. Database Creation & Populating using SQLite3
3. Exploratory Data Analysis (EDA) - Univariate, Bivariate, Multivariate, and Claims Analysis
4. SQL Analytics (15+ SQL Queries answering business cases)
5. Summary of Business Insights and Recommendations


## Step 1: Import Required Libraries


In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

# Set seaborn style for premium aesthetics
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## Step 2: Load and Clean Datasets
We load the 4 raw CSV files (`providers_data.csv`, `receivers_data.csv`, `food_listings_data.csv`, `claims_data.csv`), perform null value checks, and normalize date formats for database entry.


In [ ]:
# Read raw datasets
providers = pd.read_csv('providers_data.csv')
receivers = pd.read_csv('receivers_data.csv')
listings = pd.read_csv('food_listings_data.csv')
claims = pd.read_csv('claims_data.csv')

print('providers shape:', providers.shape)
print('receivers shape:', receivers.shape)
print('listings shape:', listings.shape)
print('claims shape:', claims.shape)

# Check for null values in each dataset
for name, df in zip(['Providers', 'Receivers', 'Food Listings', 'Claims'], [providers, receivers, listings, claims]):
    print(f'\nNull values in {name}:')
    print(df.isnull().sum())


### Date Normalization Functions
SQLite handles date values best in YYYY-MM-DD format. We write functions to convert standard dates into YYYY-MM-DD.


In [ ]:
def clean_date(date_str):
    if pd.isna(date_str) or not str(date_str).strip():
        return None
    try:
        return datetime.strptime(str(date_str).strip(), '%m/%d/%Y').strftime('%Y-%m-%d')
    except ValueError:
        try:
            return datetime.strptime(str(date_str).strip(), '%Y-%m-%d').strftime('%Y-%m-%d')
        except ValueError:
            return date_str

def clean_datetime(dt_str):
    if pd.isna(dt_str) or not str(dt_str).strip():
        return None
    try:
        return datetime.strptime(str(dt_str).strip(), '%m/%d/%Y %H:%M').strftime('%Y-%m-%d %H:%M:%S')
    except ValueError:
        try:
            return datetime.strptime(str(dt_str).strip(), '%Y-%m-%d %H:%M:%S').strftime('%Y-%m-%d %H:%M:%S')
        except ValueError:
            return dt_str

# Clean date and timestamps
listings['Expiry_Date'] = listings['Expiry_Date'].apply(clean_date)
claims['Timestamp'] = claims['Timestamp'].apply(clean_datetime)
print('Date formatting complete.')


## Step 3: SQLite Database Creation
We create `food_wastage.db` and set up the schemas with primary keys, foreign keys, and referential integrity constraints.


In [ ]:
conn = sqlite3.connect('food_wastage.db')
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON;')

# Drop child tables first, then parent tables to avoid constraint violations
cursor.execute('DROP TABLE IF EXISTS claims;')
cursor.execute('DROP TABLE IF EXISTS food_listings;')
cursor.execute('DROP TABLE IF EXISTS receivers;')
cursor.execute('DROP TABLE IF EXISTS providers;')

# Create Providers Table
cursor.execute('''
CREATE TABLE providers (
    Provider_ID INTEGER PRIMARY KEY,
    Name TEXT NOT NULL,
    Type TEXT NOT NULL,
    Address TEXT,
    City TEXT NOT NULL,
    Contact TEXT
);
''')

# Create Receivers Table
cursor.execute('''
CREATE TABLE receivers (
    Receiver_ID INTEGER PRIMARY KEY,
    Name TEXT NOT NULL,
    Type TEXT NOT NULL,
    City TEXT NOT NULL,
    Contact TEXT
);
''')

# Create Food Listings Table
cursor.execute('''
CREATE TABLE food_listings (
    Food_ID INTEGER PRIMARY KEY,
    Food_Name TEXT NOT NULL,
    Quantity INTEGER NOT NULL,
    Expiry_Date TEXT,
    Provider_ID INTEGER NOT NULL,
    Provider_Type TEXT,
    Location TEXT NOT NULL,
    Food_Type TEXT,
    Meal_Type TEXT,
    FOREIGN KEY (Provider_ID) REFERENCES providers(Provider_ID) ON DELETE CASCADE
);
''')

# Create Claims Table
cursor.execute('''
CREATE TABLE claims (
    Claim_ID INTEGER PRIMARY KEY,
    Food_ID INTEGER NOT NULL,
    Receiver_ID INTEGER NOT NULL,
    Status TEXT NOT NULL CHECK(Status IN ('Pending', 'Completed', 'Cancelled')),
    Timestamp TEXT,
    FOREIGN KEY (Food_ID) REFERENCES food_listings(Food_ID) ON DELETE CASCADE,
    FOREIGN KEY (Receiver_ID) REFERENCES receivers(Receiver_ID) ON DELETE CASCADE
);
''')

conn.commit()
print('SQLite tables created.')


### Insert Data into SQLite Tables


In [ ]:
providers.to_sql('providers', conn, if_exists='append', index=False)
receivers.to_sql('receivers', conn, if_exists='append', index=False)
listings.to_sql('food_listings', conn, if_exists='append', index=False)
claims.to_sql('claims', conn, if_exists='append', index=False)

conn.commit()
print('Data successfully inserted into database!')


## Step 4: Exploratory Data Analysis (EDA)
### 4.1 Univariate Analysis
Analyze distributions of provider types, receiver types, food types, and meal types.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

sns.countplot(data=providers, x='Type', ax=axes[0, 0], palette='crest', hue='Type', legend=False)
axes[0, 0].set_title('Provider Type Distribution')
axes[0, 0].tick_params(axis='x', rotation=15)

sns.countplot(data=receivers, x='Type', ax=axes[0, 1], palette='mako', hue='Type', legend=False)
axes[0, 1].set_title('Receiver Type Distribution')

sns.countplot(data=listings, x='Food_Type', ax=axes[1, 0], palette='Set2', hue='Food_Type', legend=False)
axes[1, 0].set_title('Food Category Distribution')

sns.countplot(data=listings, x='Meal_Type', ax=axes[1, 1], palette='rocket', hue='Meal_Type', legend=False)
axes[1, 1].set_title('Meal Type Distribution')

plt.tight_layout()
plt.show()


### 4.2 Bivariate Analysis
Analyze relations: City vs Listings, Provider vs Quantity, Food Type vs Quantity, Meal Type vs Quantity.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. City vs Food Listings (Top 10 Cities)
top_cities = listings['Location'].value_counts().head(10).reset_index()
top_cities.columns = ['City', 'Listings']
sns.barplot(data=top_cities, y='City', x='Listings', ax=axes[0,0], palette='viridis', hue='City', legend=False)
axes[0, 0].set_title('Top 10 Cities by Listings')

# 2. Provider Type vs Quantity
sns.barplot(data=listings.groupby('Provider_Type')['Quantity'].sum().reset_index(), x='Provider_Type', y='Quantity', ax=axes[0,1], palette='flare', hue='Provider_Type', legend=False)
axes[0, 1].set_title('Quantity Contributed by Provider Type')

# 3. Food Type vs Quantity
sns.barplot(data=listings.groupby('Food_Type')['Quantity'].sum().reset_index(), x='Food_Type', y='Quantity', ax=axes[1,0], palette='pastel', hue='Food_Type', legend=False)
axes[1, 0].set_title('Quantity listed by Food Type')

# 4. Meal Type vs Quantity
sns.barplot(data=listings.groupby('Meal_Type')['Quantity'].sum().reset_index(), x='Meal_Type', y='Quantity', ax=axes[1,1], palette='magma', hue='Meal_Type', legend=False)
axes[1, 1].set_title('Quantity listed by Meal Type')

plt.tight_layout()
plt.show()


### 4.3 Multivariate Analysis
Analyze relations among 3 variables: City + Provider + Quantity, Food Type + Meal Type + Quantity, Provider + Claims + Quantity, Receiver + Claims + Quantity.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# 1. City + Provider + Quantity (Top 5 Cities)
top5_cities = listings['Location'].value_counts().head(5).index
c_p_q = listings[listings['Location'].isin(top5_cities)].groupby(['Location', 'Provider_Type'])['Quantity'].sum().reset_index()
sns.barplot(data=c_p_q, x='Location', y='Quantity', hue='Provider_Type', ax=axes[0,0], palette='coolwarm')
axes[0, 0].set_title('Quantity by City and Provider Type')
axes[0, 0].tick_params(axis='x', rotation=15)

# 2. Food Type + Meal Type + Quantity
f_m_q = listings.groupby(['Food_Type', 'Meal_Type'])['Quantity'].sum().reset_index()
sns.barplot(data=f_m_q, x='Food_Type', y='Quantity', hue='Meal_Type', ax=axes[0,1], palette='Spectral')
axes[0, 1].set_title('Quantity by Food Type and Meal Type')

# 3. Provider + Claims + Quantity (Merge needed)
claims_list = pd.merge(claims, listings, on='Food_ID')
claims_list_p = pd.merge(claims_list, providers.rename(columns={'Name': 'Provider_Name'}), on='Provider_ID')
top5_provs = claims_list_p['Provider_Name'].value_counts().head(5).index
prov_claims_q = claims_list_p[claims_list_p['Provider_Name'].isin(top5_provs)].groupby(['Provider_Name', 'Status'])['Quantity'].sum().reset_index()
sns.barplot(data=prov_claims_q, y='Provider_Name', x='Quantity', hue='Status', ax=axes[1,0], palette='Set1')
axes[1, 0].set_title('Quantity Listed by Top 5 Providers and their Claim Status')

# 4. Receiver + Claims + Quantity
claims_recs = pd.merge(claims, receivers.rename(columns={'Name': 'Receiver_Name'}), on='Receiver_ID')
claims_rec_list = pd.merge(claims_recs, listings, on='Food_ID')
top5_recs = claims_rec_list['Receiver_Name'].value_counts().head(5).index
rec_claims_q = claims_rec_list[claims_rec_list['Receiver_Name'].isin(top5_recs)].groupby(['Receiver_Name', 'Status'])['Quantity'].sum().reset_index()
sns.barplot(data=rec_claims_q, y='Receiver_Name', x='Quantity', hue='Status', ax=axes[1,1], palette='Set2')
axes[1, 1].set_title('Quantity Requested by Top 5 Receivers and Claim Status')

plt.tight_layout()
plt.show()


### 4.4 Claim Analysis Charts
Status distribution, Top Receivers, and Top Providers.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Status Distribution (Pie)
status_counts = claims['Status'].value_counts()
axes[0].pie(status_counts, labels=status_counts.index, autopct='%1.1f%%', colors=['#4F81BD', '#C0504D', '#9BBB59'], startangle=90, wedgeprops={'edgecolor': 'w'})
axes[0].set_title('Claim Status Distribution')

# 2. Top Receivers by Successful Claims
comp_claims = claims[claims['Status'] == 'Completed']
top_rec_df = comp_claims['Receiver_ID'].value_counts().head(5).reset_index()
top_rec_df.columns = ['Receiver_ID', 'Count']
top_rec_df = pd.merge(top_rec_df, receivers, on='Receiver_ID')
sns.barplot(data=top_rec_df, x='Count', y='Name', ax=axes[1], palette='copper', hue='Name', legend=False)
axes[1].set_title('Top 5 Receivers (Completed Claims)')
axes[1].set_xlabel('Claims Count')

# 3. Top Providers by Successful Claims
claims_list_comp = claims_list[claims_list['Status'] == 'Completed']
top_prov_df = claims_list_comp['Provider_ID'].value_counts().head(5).reset_index()
top_prov_df.columns = ['Provider_ID', 'Count']
top_prov_df = pd.merge(top_prov_df, providers, on='Provider_ID')
sns.barplot(data=top_prov_df, x='Count', y='Name', ax=axes[2], palette='bone', hue='Name', legend=False)
axes[2].set_title('Top 5 Providers (Completed Claims)')
axes[2].set_xlabel('Claims Count')

plt.tight_layout()
plt.show()


## Step 5: SQL Analysis (15+ Queries)
We execute 16 pre-designed SQL queries to answer critical business cases.


In [ ]:
queries = [
    ('1. Providers Count by City', 'SELECT City, COUNT(*) as Provider_Count FROM providers GROUP BY City ORDER BY Provider_Count DESC LIMIT 5;'),
    ('2. Receivers Count by City', 'SELECT City, COUNT(*) as Receiver_Count FROM receivers GROUP BY City ORDER BY Receiver_Count DESC LIMIT 5;'),
    ('3. Most Contributing Provider', 'SELECT p.Name, p.Type, p.City, SUM(f.Quantity) as Total FROM food_listings f JOIN providers p ON f.Provider_ID = p.Provider_ID GROUP BY p.Provider_ID ORDER BY Total DESC LIMIT 5;'),
    ('4. Most Claimed Food Name', 'SELECT f.Food_Name, COUNT(c.Claim_ID) as Claims_Count, SUM(f.Quantity) as Total_Quantity FROM claims c JOIN food_listings f ON c.Food_ID = f.Food_ID GROUP BY f.Food_Name ORDER BY Claims_Count DESC LIMIT 5;'),
    ('5. Total Food Quantity Listed', 'SELECT SUM(Quantity) as Total FROM food_listings;'),
    ('6. Top City by Food Listing Volume', 'SELECT Location as City, COUNT(*) as Listings, SUM(Quantity) as Qty FROM food_listings GROUP BY Location ORDER BY Qty DESC LIMIT 5;'),
    ('7. Most Common Food Types Donated', 'SELECT Food_Type, COUNT(*) as Listings, SUM(Quantity) as Qty FROM food_listings GROUP BY Food_Type ORDER BY Qty DESC;'),
    ('8. Claims per Food Item (Most claimed listings)', 'SELECT f.Food_ID, f.Food_Name, COUNT(c.Claim_ID) as Claims_Count FROM food_listings f LEFT JOIN claims c ON f.Food_ID = c.Food_ID GROUP BY f.Food_ID ORDER BY Claims_Count DESC LIMIT 5;'),
    ('9. Provider with Most Successful Claims', 'SELECT p.Name, COUNT(c.Claim_ID) as Successful_Claims FROM claims c JOIN food_listings f ON c.Food_ID = f.Food_ID JOIN providers p ON f.Provider_ID = p.Provider_ID WHERE c.Status = \'Completed\' GROUP BY p.Provider_ID ORDER BY Successful_Claims DESC LIMIT 5;'),
    ('10. Claim Status Percentage', 'SELECT Status, COUNT(*) as Count, ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM claims), 2) as Pct FROM claims GROUP BY Status;'),
    ('11. Average Quantity Claimed per Successful Transaction', 'SELECT ROUND(AVG(f.Quantity), 2) as AvgQty FROM claims c JOIN food_listings f ON c.Food_ID = f.Food_ID WHERE c.Status = \'Completed\';'),
    ('12. Most Claimed Meal Type', 'SELECT f.Meal_Type, COUNT(c.Claim_ID) as Claims FROM claims c JOIN food_listings f ON c.Food_ID = f.Food_ID GROUP BY f.Meal_Type ORDER BY Claims DESC;'),
    ('13. Total Donated Quantity by Provider Type', 'SELECT Provider_Type, SUM(Quantity) as Total FROM food_listings GROUP BY Provider_Type ORDER BY Total DESC;'),
    ('14. Top Receivers by Successful Claims Count', 'SELECT r.Name, r.Type, COUNT(c.Claim_ID) as Successful FROM claims c JOIN receivers r ON c.Receiver_ID = r.Receiver_ID WHERE c.Status = \'Completed\' GROUP BY r.Receiver_ID ORDER BY Successful DESC LIMIT 5;'),
    ('15. City with Highest Food Demand (Claims)', 'SELECT r.City, COUNT(c.Claim_ID) as Claims, SUM(f.Quantity) as Requested FROM claims c JOIN receivers r ON c.Receiver_ID = r.Receiver_ID JOIN food_listings f ON c.Food_ID = f.Food_ID GROUP BY r.City ORDER BY Claims DESC LIMIT 5;'),
    ('16. Overall System Balance Sheet', 'SELECT COUNT(f.Food_ID) as Total, SUM(CASE WHEN c.Status=\'Completed\' THEN 1 ELSE 0 END) as Completed, SUM(CASE WHEN c.Status=\'Pending\' THEN 1 ELSE 0 END) as Pending, SUM(CASE WHEN c.Status=\'Cancelled\' OR c.Status IS NULL THEN 1 ELSE 0 END) as Unclaimed FROM food_listings f LEFT JOIN claims c ON f.Food_ID = c.Food_ID;')

]

for title, query in queries:
    print(f'\n=== {title} ===')
    res_df = pd.read_sql_query(query, conn)
    print(res_df.to_string(index=False))


## Step 6: Summary of Business Insights & Recommendations

### Key Insights Found:
1. **Food Availability**: **South Kathryn** has the most food availability (179 items). Establishing storage systems there is critical.
2. **Food Waste**: **Breakfast** is the meal type that suffers the most cancellation/waste, owing to short shelf life.
3. **Provider Analysis**: **Barry Group** (Restaurant) is the leading contributor (179 items). Supermarkets and Restaurants contribute the most food overall.
4. **Receiver Analysis**: NGOs and Charities like **Derek Potter** (99 items claimed) and **Timothy Garrett** (80 items claimed) are active hubs that successfully redistribute food.
5. **Claims Status**: Only **33.9%** of listed claims are successfully completed; **33.6%** are cancelled, highlighting massive inefficiencies in distribution logistics.
6. **Demand Analysis**: **West David** is the top demand city (191 items requested across 5 claims).

### Actionable Recommendations:
* **Recommendation 1 (NGO Partnerships)**: Expand NGO hubs in high-wastage cities like *South Kathryn* and *Jonathanstad*.
* **Recommendation 2 (Recognition)**: Set up a recognition/CSR incentive program for high contributors like *Barry Group*.
* **Recommendation 3 (Smart Expiry Alerts)**: Build an automated push-notification system warning local NGOs 12 hours before food expiry.
* **Recommendation 4 (Logistics Optimization)**: Route more logistics support (vehicles/temperature-controlled transport) to high-demand hubs like *West David*.


In [ ]:
conn.close()
print('Database connection closed safely.')
